# 2.1 Data Preparation

What this section is about:

1. preprocess the raw `binSamples` and `fillSamples` so later fitting and simulation code can traverse the data efficiently;
2. split the data into in-sample and out-of-sample sets;
3. decide which datasets should be merged and which should remain separate;
4. implement the baseline: one in-sample month, next out-of-sample month, same 20 stocks;
5. implement the enhancement: rolling month-by-month structure over the whole available stock universe.



## 0. Preliminaries

Run these cells first. They install/load the packages and copy the raw files from Google Drive to Colab local storage. Working locally inside `/content/project_data` is much faster than repeatedly reading large CSV files from Drive.


In [33]:
!pip -q install pyarrow

import os
import re
import gc
import json
import shutil
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)


In [34]:
# checking if the file is on drive

from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [35]:
# Raw data location in Drive.
DRIVE_BASE = Path("/content/drive/MyDrive/Quantitative Trading and Price Impact")
BIN_DRIVE = DRIVE_BASE / "binSamples"
FILL_DRIVE = DRIVE_BASE / "fillSamples"

# Local working location in google colab.
LOCAL_BASE = Path("/content/project_data")
BIN_LOCAL = LOCAL_BASE / "binSamples"
FILL_LOCAL = LOCAL_BASE / "fillSamples"

PROCESSED_BASE = LOCAL_BASE / "processed_2_1"
MONTHLY_DIR = PROCESSED_BASE / "monthly_standardised"
BASELINE_DIR = PROCESSED_BASE / "baseline_20stocks"
ROLLING_DIR = PROCESSED_BASE / "rolling_full_universe"
REPORT_DIR = PROCESSED_BASE / "report_tables"

for p in [LOCAL_BASE, BIN_LOCAL, FILL_LOCAL, PROCESSED_BASE, MONTHLY_DIR, BASELINE_DIR, ROLLING_DIR, REPORT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Drive base:       ", DRIVE_BASE)
print("Local raw base:   ", LOCAL_BASE)
print("Processed outputs:", PROCESSED_BASE)


Drive base:        /content/drive/MyDrive/Quantitative Trading and Price Impact
Local raw base:    /content/project_data
Processed outputs: /content/project_data/processed_2_1


In [36]:
# Copy from Drive to local Colab disk.
!rsync -ah --progress "/content/drive/MyDrive/Quantitative Trading and Price Impact/binSamples/" "/content/project_data/binSamples/"
!rsync -ah --progress "/content/drive/MyDrive/Quantitative Trading and Price Impact/fillSamples/" "/content/project_data/fillSamples/"

sending incremental file list
sending incremental file list


## 1. Locate files and verify the raw layout

The project data has 12 monthly files in `binSamples` and 12 monthly files in `fillSamples`. We identify the month from filenames such as `bin201901.csv` and `fills201901.csv`.


In [37]:
def month_from_filename(path: Path) -> str:
    """Extract YYYYMM from a filename such as bin201901.csv or fills201901.csv."""
    m = re.search(r"(20\d{4})", path.name)
    if m is None:
        raise ValueError(f"Could not extract month from filename: {path}")
    return m.group(1)


def list_csv_like_files(folder: Path) -> List[Path]:
    """Return csv/csv-like files, ignoring hidden/system files."""
    if not folder.exists():
        return []
    files = []
    for p in folder.iterdir():
        if p.name.startswith("."):
            continue
        if p.is_file() and p.suffix.lower() in [".csv", ".txt", ""]:
            files.append(p)
    return sorted(files, key=lambda x: month_from_filename(x))

bin_files = list_csv_like_files(BIN_LOCAL)
fill_files = list_csv_like_files(FILL_LOCAL)

assert len(bin_files) > 0, "No binSamples files found. Check the Drive/local path."
assert len(fill_files) > 0, "No fillSamples files found. Check the Drive/local path."


In [38]:
# Build a file registry table.
file_registry = []
for kind, files in [("bin", bin_files), ("fills", fill_files)]:
    for p in files:
        file_registry.append({
            "kind": kind,
            "month": month_from_filename(p),
            "file_name": p.name,
            "path": str(p),
            "size_mb": p.stat().st_size / 1024**2,
        })

file_registry = pd.DataFrame(file_registry).sort_values(["kind", "month"]).reset_index(drop=True)
display(file_registry)

file_registry.to_csv(REPORT_DIR / "raw_file_registry.csv", index=False)


,kind,month,file_name,path,size_mb
0,bin,201901,bin201901.csv,/content/project_data/binSamples/bin201901.csv,195.925348
1,bin,201902,bin201902.csv,/content/project_data/binSamples/bin201902.csv,182.357881
2,bin,201903,bin201903.csv,/content/project_data/binSamples/bin201903.csv,205.152733
3,bin,201904,bin201904.csv,/content/project_data/binSamples/bin201904.csv,196.481595
4,bin,201905,bin201905.csv,/content/project_data/binSamples/bin201905.csv,215.016769
5,bin,201906,bin201906.csv,/content/project_data/binSamples/bin201906.csv,190.156547
6,bin,201907,bin201907.csv,/content/project_data/binSamples/bin201907.csv,202.883371
7,bin,201908,bin201908.csv,/content/project_data/binSamples/bin201908.csv,213.513826
8,bin,201909,bin201909.csv,/content/project_data/binSamples/bin201909.csv,182.231578
9,bin,201910,bin201910.csv,/content/project_data/binSamples/bin201910.csv,208.273834


## 2. Schema checks

Before preprocessing, confirm that each monthly file has the expected columns. This protects later code from silently breaking if one file has a slightly different format.


In [39]:
EXPECTED_BIN_COLS = [
    "date", "time", "stock", "trade", "orderFlow", "hidden", "auction",
    "mid", "midEnd", "spread", "effSpread", "lobImb", "effLobImb",
    "trdLiq", "ofLiq", "depth", "nbEvents", "nbHidden", "nbTrades"
]

EXPECTED_FILL_COLS = [
    "date", "stock", "time", "trade", "mid", "spread", "effSpread", "depth",
    "lobImb", "ask", "bid", "askVolume", "bidVolume"
]


def read_columns(path: Path) -> List[str]:
    return pd.read_csv(path, nrows=0).columns.tolist()

schema_rows = []
for p in bin_files:
    cols = read_columns(p)
    schema_rows.append({
        "kind": "bin",
        "month": month_from_filename(p),
        "file_name": p.name,
        "n_cols": len(cols),
        "matches_expected": cols == EXPECTED_BIN_COLS,
        "missing_expected_cols": sorted(set(EXPECTED_BIN_COLS) - set(cols)),
        "extra_cols": sorted(set(cols) - set(EXPECTED_BIN_COLS)),
    })

for p in fill_files:
    cols = read_columns(p)
    schema_rows.append({
        "kind": "fills",
        "month": month_from_filename(p),
        "file_name": p.name,
        "n_cols": len(cols),
        "matches_expected": cols == EXPECTED_FILL_COLS,
        "missing_expected_cols": sorted(set(EXPECTED_FILL_COLS) - set(cols)),
        "extra_cols": sorted(set(cols) - set(EXPECTED_FILL_COLS)),
    })

schema_check = pd.DataFrame(schema_rows).sort_values(["kind", "month"]).reset_index(drop=True)
display(schema_check)
schema_check.to_csv(REPORT_DIR / "schema_check.csv", index=False)

if not schema_check["matches_expected"].all():
    print("WARNING: At least one file has a non-standard schema. Inspect schema_check before continuing.")


,kind,month,file_name,n_cols,matches_expected,missing_expected_cols,extra_cols
0,bin,201901,bin201901.csv,19,True,[],[]
1,bin,201902,bin201902.csv,19,True,[],[]
2,bin,201903,bin201903.csv,19,True,[],[]
3,bin,201904,bin201904.csv,19,True,[],[]
4,bin,201905,bin201905.csv,19,True,[],[]
5,bin,201906,bin201906.csv,19,True,[],[]
6,bin,201907,bin201907.csv,19,True,[],[]
7,bin,201908,bin201908.csv,19,True,[],[]
8,bin,201909,bin201909.csv,19,True,[],[]
9,bin,201910,bin201910.csv,19,True,[],[]


In [40]:
# Show small examples from the first bin and fill files.
sample_bin = pd.read_csv(bin_files[0], nrows=5)
sample_fill = pd.read_csv(fill_files[0], nrows=5)

print("Sample bin rows:")
display(sample_bin)
print("Sample fill rows:")
display(sample_fill)


Sample bin rows:


,date,time,stock,trade,orderFlow,hidden,auction,mid,midEnd,spread,effSpread,lobImb,effLobImb,trdLiq,ofLiq,depth,nbEvents,nbHidden,nbTrades
0,2019-01-02,09:30:10,A,0,1300,100,0,66.215,66.220,0.285,NaN,-0.333333,NaN,0,1300,300,2,1,0
1,2019-01-02,09:30:20,A,0,-300,100,0,66.335,66.335,0.145,NaN,0.333333,NaN,0,300,350,2,1,0
2,2019-01-02,09:30:30,A,0,200,0,0,66.335,66.335,0.145,NaN,0.000000,NaN,0,200,600,1,0,0
3,2019-01-02,09:30:40,A,0,300,0,0,66.340,66.340,0.140,NaN,0.200000,NaN,0,300,500,2,0,0
4,2019-01-02,09:31:00,A,0,0,100,0,66.270,66.270,0.070,NaN,0.500000,NaN,0,400,375,4,1,0


Sample fill rows:


,date,stock,time,trade,mid,spread,effSpread,depth,lobImb,ask,bid,askVolume,bidVolume
0,2019-01-02,AAPL,09:30:00.142,-70,154.770,0.160,-2.024990,405,-0.506173,154.93,154.61,305,100
1,2019-01-02,AAPL,09:30:00.185,100,154.740,0.140,2.026321,223,-0.838565,154.88,154.60,205,18
2,2019-01-02,AAPL,09:30:00.210,-100,154.740,0.140,-2.027649,323,-0.888545,154.88,154.60,305,18
3,2019-01-02,AAPL,09:30:00.426,-234,154.745,0.135,-2.028056,371,-0.644205,154.88,154.61,305,66
4,2019-01-02,AAPL,09:30:00.429,-100,154.705,0.105,-2.027588,289,-0.418685,154.81,154.60,205,84


## 3. Preprocessing functions

The preprocessing is deliberately tied to Section 2.1:

- create a proper `datetime` column from `date` and `time`;
- create `trading_date` and `seconds_from_open` for efficient intraday traversal;
- keep valid rows with usable stock names and positive mid prices;
- add simple notional/volume variables needed later;
- for `binSamples`, add within-bin price movement variables from `mid` to `midEnd`;
- save standardised monthly files as compressed Parquet.

No model fitting is done here.


In [41]:
# dictionary of column names as keys and their types as values respectively

BIN_DTYPES = {
    "date": "string",
    "time": "string",
    "stock": "string",
    "trade": "float64",
    "orderFlow": "float64",
    "hidden": "float64",
    "auction": "float64",
    "mid": "float64",
    "midEnd": "float64",
    "spread": "float64",
    "effSpread": "float64",
    "lobImb": "float64",
    "effLobImb": "float64",
    "trdLiq": "float64",
    "ofLiq": "float64",
    "depth": "float64",
    "nbEvents": "float64",
    "nbHidden": "float64",
    "nbTrades": "float64",
}

FILL_DTYPES = {
    "date": "string",
    "time": "string",
    "stock": "string",
    "trade": "float64",
    "mid": "float64",
    "spread": "float64",
    "effSpread": "float64",
    "depth": "float64",
    "lobImb": "float64",
    "ask": "float64",
    "bid": "float64",
    "askVolume": "float64",
    "bidVolume": "float64",
}


def read_raw_month(path: Path, kind: str) -> pd.DataFrame:
    """Read one raw monthly file with consistent dtypes."""
    if kind == "bin":
        return pd.read_csv(path, dtype=BIN_DTYPES)
    elif kind == "fills":
        return pd.read_csv(path, dtype=FILL_DTYPES)
    else:
        raise ValueError("kind must be 'bin' or 'fills'")


def standardise_intraday_data(df: pd.DataFrame, *, kind: str, month: str, source_file: str) -> pd.DataFrame:
    """Standardise one raw monthly dataframe for efficient downstream use."""
    df = df.copy()

    # Basic identifiers.
    df["month"] = month
    df["source_file"] = source_file
    df["stock"] = df["stock"].astype("string").str.strip()

    # Robust datetime parsing. fillSamples has milliseconds; binSamples usually has 10-second bins.
    df["datetime"] = pd.to_datetime(df["date"].astype(str) + " " + df["time"].astype(str), errors="coerce")
    df["trading_date"] = pd.to_datetime(df["date"], errors="coerce").dt.date.astype("string")

    # Seconds since 09:30:00. This is useful for intraday loops and daily resets later.
    session_open = pd.to_timedelta("09:30:00")
    time_of_day = df["datetime"] - df["datetime"].dt.normalize()
    df["seconds_from_open"] = (time_of_day - session_open).dt.total_seconds().astype("float64")

    # Keep only rows with essential fields. Do not over-clean: later sections may need zeros in trade/orderFlow.
    before = len(df)
    df = df[df["datetime"].notna()]
    df = df[df["stock"].notna() & (df["stock"] != "")]
    df = df[df["mid"].notna() & (df["mid"] > 0)]
    after = len(df)

    # Core trading variables.
    df["abs_trade"] = df["trade"].abs()
    df["signed_notional"] = df["trade"] * df["mid"]
    df["abs_notional"] = df["abs_trade"] * df["mid"]

    # Spread/depth diagnostics. These are useful report and sanity-check variables.
    df["spread_bps"] = np.where(df["mid"] > 0, 10000.0 * df["spread"] / df["mid"], np.nan)

    if kind == "bin":
        df["abs_order_flow"] = df["orderFlow"].abs()
        df["signed_of_notional"] = df["orderFlow"] * df["mid"]
        df["abs_of_notional"] = df["abs_order_flow"] * df["mid"]

    # Sorting once here makes later grouped traversal faster and less error-prone.
    df = df.sort_values(["stock", "datetime"]).reset_index(drop=True)

    # Use category for repeated identifiers to reduce memory/parquet size.
    for col in ["stock", "month", "source_file", "trading_date"]:
        if col in df.columns:
            df[col] = df[col].astype("category")

    df.attrs["n_rows_dropped_basic_cleaning"] = int(before - after)
    return df


def monthly_stock_stats(df: pd.DataFrame, *, kind: str) -> pd.DataFrame:
    """Create stock-level monthly summary used for universe selection and report tables."""
    agg_dict = {
        "datetime": ["min", "max", "count"],
        "trading_date": pd.Series.nunique,
        "abs_trade": "sum",
        "abs_notional": "sum",
        "mid": ["mean", "median"],
        "spread_bps": "median",
        "depth": "median",
    }
    if kind == "bin":
        agg_dict.update({
            "abs_order_flow": "sum",
            "abs_of_notional": "sum",
            "nbTrades": "sum",
            "nbEvents": "sum",
        })

    out = df.groupby("stock", observed=True).agg(agg_dict)
    out.columns = ["_".join([str(x) for x in col if str(x) != ""]) for col in out.columns]
    out = out.reset_index()
    out = out.rename(columns={
        "datetime_min": "first_datetime",
        "datetime_max": "last_datetime",
        "datetime_count": "n_rows",
        "trading_date_nunique": "n_trading_days",
        "abs_trade_sum": "total_abs_trade",
        "abs_notional_sum": "total_abs_notional",
        "mid_mean": "mean_mid",
        "mid_median": "median_mid",
        "spread_bps_median": "median_spread_bps",
        "depth_median": "median_depth",
    })
    return out


def save_parquet(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, index=False, compression="zstd")


## 4. Process all monthly files into standardised monthly Parquet files

This is the key preprocessing step for efficient traversal. Instead of repeatedly reading the large raw CSVs, later sections can read these standardised Parquet files.


In [42]:
# Set this to False only if you already ran this cell and want to skip re-processing.
RUN_MONTHLY_STANDARDISATION = True

processed_rows = []
all_bin_stock_stats = []
all_fill_stock_stats = []

if RUN_MONTHLY_STANDARDISATION:
    # Process binSamples.
    for raw_path in bin_files:
        month = month_from_filename(raw_path)
        out_path = MONTHLY_DIR / "bin" / f"bin_{month}_standardised.parquet"
        stats_path = MONTHLY_DIR / "bin" / f"bin_{month}_stock_stats.csv"

        print(f"Processing bin {month}: {raw_path.name}")
        raw = read_raw_month(raw_path, kind="bin")
        clean = standardise_intraday_data(raw, kind="bin", month=month, source_file=raw_path.name)
        stats = monthly_stock_stats(clean, kind="bin")
        stats["month"] = month
        stats["parquet_path"] = str(out_path)
        stats["raw_file"] = raw_path.name

        save_parquet(clean, out_path)
        stats.to_csv(stats_path, index=False)

        processed_rows.append({
            "kind": "bin",
            "month": month,
            "raw_file": raw_path.name,
            "raw_size_mb": raw_path.stat().st_size / 1024**2,
            "n_raw_rows": len(raw),
            "n_clean_rows": len(clean),
            "n_rows_dropped_basic_cleaning": clean.attrs.get("n_rows_dropped_basic_cleaning", np.nan),
            "n_stocks": clean["stock"].nunique(),
            "first_datetime": clean["datetime"].min(),
            "last_datetime": clean["datetime"].max(),
            "parquet_path": str(out_path),
            "parquet_size_mb": out_path.stat().st_size / 1024**2,
        })
        all_bin_stock_stats.append(stats)
        del raw, clean, stats
        gc.collect()


    processed_summary = pd.DataFrame(processed_rows).sort_values(["kind", "month"]).reset_index(drop=True)
    processed_summary.to_csv(REPORT_DIR / "processed_monthly_summary.csv", index=False)

    bin_stock_month_stats = pd.concat(all_bin_stock_stats, ignore_index=True) if all_bin_stock_stats else pd.DataFrame()
    fill_stock_month_stats = pd.concat(all_fill_stock_stats, ignore_index=True) if all_fill_stock_stats else pd.DataFrame()
    bin_stock_month_stats.to_csv(REPORT_DIR / "bin_stock_month_stats.csv", index=False)
    fill_stock_month_stats.to_csv(REPORT_DIR / "fill_stock_month_stats.csv", index=False)

else:
    processed_summary = pd.read_csv(REPORT_DIR / "processed_monthly_summary.csv")
    bin_stock_month_stats = pd.read_csv(REPORT_DIR / "bin_stock_month_stats.csv")
    fill_stock_month_stats = pd.read_csv(REPORT_DIR / "fill_stock_month_stats.csv")

print("Processed monthly summary:")
display(processed_summary)
bin_months = sorted(bin_stock_month_stats["month"].astype(str).unique().tolist())
fill_months = []



Processing bin 201901: bin201901.csv
Processing bin 201902: bin201902.csv
Processing bin 201903: bin201903.csv
Processing bin 201904: bin201904.csv


KeyboardInterrupt: 

## 5. Decide what to merge and what to keep separate

This is explicitly required by Section 2.1. The decision is:

- keep `binSamples` and `fillSamples` separate because they have different granularity and different schemas;
- standardise each month and save it as Parquet;
- for the baseline, create exact 20-stock train/test files because later model fitting will repeatedly use them;
- for the rolling enhancement, do **not** duplicate huge train/test files for every pair. Instead, keep one standardised Parquet per month and create rolling pair/universe manifests. This is faster and cleaner.


In [18]:
storage_plan = pd.DataFrame([
    {
        "dataset": "binSamples monthly files",
        "action": "standardise and save one Parquet per month",
        "reason": "Main public binned tape for impact-model fitting; monthly Parquets are faster than raw CSV and preserve chronological traversal.",
    },
    {
        "dataset": "fillSamples monthly files",
        "action": "standardise and save one Parquet per month, but keep separate from binSamples",
        "reason": "Different schema and event-level/fill-level granularity; direct merging with binSamples would create unnecessary duplication and alignment problems.",
    },
    {
        "dataset": "baseline train/test",
        "action": "materialise 20-stock train and test Parquets",
        "reason": "The baseline repeatedly uses exactly the same 20 stocks over two months, so a small materialised subset speeds up Sections 2.2 and 2.3.",
    },
    {
        "dataset": "rolling enhancement",
        "action": "use monthly Parquets plus rolling manifests, not duplicated rolling Parquets",
        "reason": "Rolling over the whole universe would duplicate large files for every consecutive month pair; a manifest tells later code which monthly files and stocks to load.",
    },
])

display(storage_plan)
storage_plan.to_csv(REPORT_DIR / "merge_vs_separate_decision.csv", index=False)


,dataset,action,reason
0,binSamples monthly files,standardise and save one Parquet per month,Main public binned tape for impact-model fitti...
1,fillSamples monthly files,"standardise and save one Parquet per month, bu...",Different schema and event-level/fill-level gr...
2,baseline train/test,materialise 20-stock train and test Parquets,The baseline repeatedly uses exactly the same ...
3,rolling enhancement,"use monthly Parquets plus rolling manifests, n...",Rolling over the whole universe would duplicat...


## 6. Baseline split: one in-sample month, next out-of-sample month, same 20 stocks

Baseline rule implemented here:

- train/in-sample month = first available month, normally `201901`;
- test/out-of-sample month = next available month, normally `201902`;
- select 20 stocks that are present in both months;
- rank candidates by in-sample traded notional, so the chosen stocks are liquid and usable for fitting/simulation;
- save baseline train/test Parquet files and a config JSON.


In [19]:
def parquet_path_for(kind: str, month: str) -> Path:
    if kind == "bin":
        return MONTHLY_DIR / "bin" / f"bin_{month}_standardised.parquet"
    elif kind == "fills":
        return MONTHLY_DIR / "fills" / f"fills_{month}_standardised.parquet"
    else:
        raise ValueError("kind must be 'bin' or 'fills'")


def select_baseline_20_stocks(
    train_month: str,
    test_month: str,
    stats: pd.DataFrame,
    n_stocks: int = 20,
) -> List[str]:
    """Select the top n liquid stocks present in both train and test month."""
    s = stats.copy()
    s["month"] = s["month"].astype(str)
    train = s[s["month"] == str(train_month)].copy()
    test = s[s["month"] == str(test_month)].copy()

    train_stocks = set(train["stock"].astype(str))
    test_stocks = set(test["stock"].astype(str))
    common = train_stocks & test_stocks

    if len(common) < n_stocks:
        raise ValueError(f"Only {len(common)} stocks are common to {train_month} and {test_month}; need {n_stocks}.")

    selected = (
        train[train["stock"].astype(str).isin(common)]
        .sort_values("total_abs_notional", ascending=False)
        .head(n_stocks)["stock"]
        .astype(str)
        .tolist()
    )
    return selected


def load_month_subset(kind: str, month: str, stocks: Optional[List[str]] = None) -> pd.DataFrame:
    path = parquet_path_for(kind, month)
    df = pd.read_parquet(path)
    if stocks is not None:
        df = df[df["stock"].astype(str).isin(stocks)].copy()
    return df.sort_values(["stock", "datetime"]).reset_index(drop=True)


In [20]:
# Baseline months. By default: first two processed bin months.
# Change these manually only if your group wants a different pair.
BASELINE_TRAIN_MONTH = bin_months[0]
BASELINE_TEST_MONTH = bin_months[1]
N_BASELINE_STOCKS = 20

baseline_stocks = select_baseline_20_stocks(
    BASELINE_TRAIN_MONTH,
    BASELINE_TEST_MONTH,
    bin_stock_month_stats,
    n_stocks=N_BASELINE_STOCKS,
)

print("Baseline train month:", BASELINE_TRAIN_MONTH)
print("Baseline test month: ", BASELINE_TEST_MONTH)
print("Selected 20 stocks: ", baseline_stocks)


Baseline train month: 201901
Baseline test month:  201902
Selected 20 stocks:  ['AMZN', 'AAPL', 'AMD', 'AMGN', 'ADBE', 'ALGN', 'AMAT', 'ANTM', 'AAL', 'ADP', 'ABBV', 'ADI', 'ABT', 'ADSK', 'ALXN', 'AGN', 'ACN', 'AAP', 'AMT', 'APD']


In [21]:
# Load, filter, and save baseline binSamples.
bin_train_20 = load_month_subset("bin", BASELINE_TRAIN_MONTH, baseline_stocks)
bin_test_20 = load_month_subset("bin", BASELINE_TEST_MONTH, baseline_stocks)

assert sorted(bin_train_20["stock"].astype(str).unique()) == sorted(baseline_stocks)
assert sorted(bin_test_20["stock"].astype(str).unique()) == sorted(baseline_stocks)

baseline_bin_train_path = BASELINE_DIR / f"bin_train_{BASELINE_TRAIN_MONTH}_20stocks.parquet"
baseline_bin_test_path = BASELINE_DIR / f"bin_test_{BASELINE_TEST_MONTH}_20stocks.parquet"
save_parquet(bin_train_20, baseline_bin_train_path)
save_parquet(bin_test_20, baseline_bin_test_path)

print("Saved:", baseline_bin_train_path)
print("Saved:", baseline_bin_test_path)
print("Train shape:", bin_train_20.shape)
print("Test shape: ", bin_test_20.shape)


Saved: /content/project_data/processed_2_1/baseline_20stocks/bin_train_201901_20stocks.parquet
Saved: /content/project_data/processed_2_1/baseline_20stocks/bin_test_201902_20stocks.parquet
Train shape: (901172, 34)
Test shape:  (846332, 34)


In [22]:
# Prepare fillSamples for the same two baseline months.
# We do NOT force fillSamples to have the same 20-stock universe, because in this dataset fills may cover fewer stocks.
# We simply filter to baseline stocks when they exist and save the available rows.

fill_train_path = parquet_path_for("fills", BASELINE_TRAIN_MONTH) if BASELINE_TRAIN_MONTH in fill_months else None
fill_test_path = parquet_path_for("fills", BASELINE_TEST_MONTH) if BASELINE_TEST_MONTH in fill_months else None

baseline_fill_outputs = {}

if fill_train_path is not None and fill_train_path.exists():
    fills_train = load_month_subset("fills", BASELINE_TRAIN_MONTH, baseline_stocks)
    out = BASELINE_DIR / f"fills_train_{BASELINE_TRAIN_MONTH}_available_baseline_stocks.parquet"
    save_parquet(fills_train, out)
    baseline_fill_outputs["fills_train_path"] = str(out)
    print("Saved:", out, "shape:", fills_train.shape, "stocks:", sorted(fills_train["stock"].astype(str).unique()))
else:
    fills_train = pd.DataFrame()
    baseline_fill_outputs["fills_train_path"] = None
    print("No fill file available for train month.")

if fill_test_path is not None and fill_test_path.exists():
    fills_test = load_month_subset("fills", BASELINE_TEST_MONTH, baseline_stocks)
    out = BASELINE_DIR / f"fills_test_{BASELINE_TEST_MONTH}_available_baseline_stocks.parquet"
    save_parquet(fills_test, out)
    baseline_fill_outputs["fills_test_path"] = str(out)
    print("Saved:", out, "shape:", fills_test.shape, "stocks:", sorted(fills_test["stock"].astype(str).unique()))
else:
    fills_test = pd.DataFrame()
    baseline_fill_outputs["fills_test_path"] = None
    print("No fill file available for test month.")


Saved: /content/project_data/processed_2_1/baseline_20stocks/fills_train_201901_available_baseline_stocks.parquet shape: (593243, 23) stocks: ['AAPL']
Saved: /content/project_data/processed_2_1/baseline_20stocks/fills_test_201902_available_baseline_stocks.parquet shape: (375224, 23) stocks: ['AAPL']


In [23]:
# Baseline summary table for the report.
baseline_stats = bin_stock_month_stats[
    (bin_stock_month_stats["month"].astype(str).isin([BASELINE_TRAIN_MONTH, BASELINE_TEST_MONTH]))
    & (bin_stock_month_stats["stock"].astype(str).isin(baseline_stocks))
].copy()

baseline_stats = baseline_stats[[
    "month", "stock", "n_rows", "n_trading_days", "total_abs_trade", "total_abs_notional",
    "mean_mid", "median_spread_bps", "median_depth"
]].sort_values(["month", "total_abs_notional"], ascending=[True, False])

baseline_stats.to_csv(BASELINE_DIR / "baseline_20stocks_summary.csv", index=False)
display(baseline_stats.head(40))


,month,stock,n_rows,n_trading_days,total_abs_trade,total_abs_notional,mean_mid,median_spread_bps,median_depth
38,201901,AMZN,46814,21,13373097.0,2.185694e+10,1637.486927,1.563794,116.57140
3,201901,AAPL,46860,21,62337719.0,9.531554e+09,153.935871,0.638244,728.33330
32,201901,AMD,46858,21,194059514.0,3.953044e+09,20.266157,2.473411,10767.88000
35,201901,AMGN,46459,21,11388381.0,2.225017e+09,196.554986,1.790877,328.20000
9,201901,ADBE,46392,21,8701685.0,2.067497e+09,237.649651,2.282442,261.60000
26,201901,ALGN,42508,21,5400709.0,1.139455e+09,208.893453,5.469836,242.33330
31,201901,AMAT,46844,21,27984501.0,9.931631e+08,35.379268,1.458364,1700.00000
41,201901,ANTM,42647,21,3221600.0,8.661784e+08,263.781471,3.004243,191.25000
1,201901,AAL,46782,21,25971169.0,8.600217e+08,33.267361,1.550147,1228.36650
12,201901,ADP,45534,21,6382577.0,8.534474e+08,133.064680,1.907159,305.50000


In [24]:
# Save full baseline configuration.
baseline_config = {
    "section": "2.1 Data Preparation",
    "baseline_train_month": BASELINE_TRAIN_MONTH,
    "baseline_test_month": BASELINE_TEST_MONTH,
    "n_baseline_stocks": N_BASELINE_STOCKS,
    "baseline_stocks": baseline_stocks,
    "selection_rule": "Top 20 stocks by in-sample total_abs_notional among stocks present in both train and test months.",
    "bin_and_fills_merged": False,
    "reason_not_merged": "binSamples and fillSamples have different granularity and schemas; they are saved separately.",
}

with open(BASELINE_DIR / "section_2_1_baseline_config.json", "w") as f:
    json.dump(baseline_config, f, indent=2)

print(json.dumps(baseline_config, indent=2))


{
  "section": "2.1 Data Preparation",
  "baseline_train_month": "201901",
  "baseline_test_month": "201902",
  "n_baseline_stocks": 20,
  "baseline_stocks": [
    "AMZN",
    "AAPL",
    "AMD",
    "AMGN",
    "ADBE",
    "ALGN",
    "AMAT",
    "ANTM",
    "AAL",
    "ADP",
    "ABBV",
    "ADI",
    "ABT",
    "ADSK",
    "ALXN",
    "AGN",
    "ACN",
    "AAP",
    "AMT",
    "APD"
  ],
  "bin_train_path": "/content/project_data/processed_2_1/baseline_20stocks/bin_train_201901_20stocks.parquet",
  "bin_test_path": "/content/project_data/processed_2_1/baseline_20stocks/bin_test_201902_20stocks.parquet",
  "fills_train_path": "/content/project_data/processed_2_1/baseline_20stocks/fills_train_201901_available_baseline_stocks.parquet",
  "fills_test_path": "/content/project_data/processed_2_1/baseline_20stocks/fills_test_201902_available_baseline_stocks.parquet",
  "selection_rule": "Top 20 stocks by in-sample total_abs_notional among stocks present in both train and test months.",
  "

## 7. Enhancement: rolling month pairs over the whole stock universe

For the enhancement, we use every consecutive month pair:

`201901 -> 201902`, `201902 -> 201903`, ..., `201911 -> 201912`.

For each pair, the universe is **all stocks present in both months of that pair**. This avoids the baseline restriction of exactly 20 stocks.

To keep this efficient, the rolling enhancement saves manifests instead of duplicating large train/test datasets repeatedly.


In [25]:
def build_consecutive_month_pairs(months: List[str]) -> List[Tuple[str, str]]:
    months = sorted([str(m) for m in months])
    return list(zip(months[:-1], months[1:]))

rolling_pairs = build_consecutive_month_pairs(bin_months)
print("Rolling pairs:")
for tr, te in rolling_pairs:
    print(f"  {tr} -> {te}")

assert len(rolling_pairs) >= 1, "Need at least two months for rolling train/test pairs."


Rolling pairs:
  201901 -> 201902
  201902 -> 201903
  201903 -> 201904
  201904 -> 201905
  201905 -> 201906
  201906 -> 201907
  201907 -> 201908
  201908 -> 201909
  201909 -> 201910
  201910 -> 201911
  201911 -> 201912


In [26]:
# Build rolling pair summary and universe tables.
rolling_pair_rows = []
rolling_universe_rows = []

stats = bin_stock_month_stats.copy()
stats["month"] = stats["month"].astype(str)
stats["stock"] = stats["stock"].astype(str)

for pair_id, (train_month, test_month) in enumerate(rolling_pairs, start=1):
    train_stats = stats[stats["month"] == train_month].copy()
    test_stats = stats[stats["month"] == test_month].copy()

    train_stocks = set(train_stats["stock"])
    test_stocks = set(test_stats["stock"])
    common_stocks = sorted(train_stocks & test_stocks)

    # Merge stock-level metrics for the common universe.
    pair_universe = (
        train_stats[train_stats["stock"].isin(common_stocks)][["stock", "total_abs_notional", "n_rows", "n_trading_days"]]
        .rename(columns={
            "total_abs_notional": "train_total_abs_notional",
            "n_rows": "train_n_rows",
            "n_trading_days": "train_n_trading_days",
        })
        .merge(
            test_stats[test_stats["stock"].isin(common_stocks)][["stock", "total_abs_notional", "n_rows", "n_trading_days"]]
            .rename(columns={
                "total_abs_notional": "test_total_abs_notional",
                "n_rows": "test_n_rows",
                "n_trading_days": "test_n_trading_days",
            }),
            on="stock",
            how="inner",
        )
    )
    pair_universe["pair_id"] = pair_id
    pair_universe["train_month"] = train_month
    pair_universe["test_month"] = test_month
    pair_universe = pair_universe.sort_values("train_total_abs_notional", ascending=False)
    rolling_universe_rows.append(pair_universe)

    rolling_pair_rows.append({
        "pair_id": pair_id,
        "train_month": train_month,
        "test_month": test_month,
        "train_bin_path": str(parquet_path_for("bin", train_month)),
        "test_bin_path": str(parquet_path_for("bin", test_month)),
        "train_fill_path": str(parquet_path_for("fills", train_month)) if train_month in fill_months else None,
        "test_fill_path": str(parquet_path_for("fills", test_month)) if test_month in fill_months else None,
        "n_train_stocks": len(train_stocks),
        "n_test_stocks": len(test_stocks),
        "n_common_stocks_pair_universe": len(common_stocks),
        "top20_common_stocks_by_train_notional": pair_universe.head(20)["stock"].tolist(),
    })

rolling_pair_summary = pd.DataFrame(rolling_pair_rows)
rolling_universe_by_pair = pd.concat(rolling_universe_rows, ignore_index=True)

rolling_pair_summary.to_csv(ROLLING_DIR / "rolling_pair_summary.csv", index=False)
rolling_universe_by_pair.to_csv(ROLLING_DIR / "rolling_universe_by_pair.csv", index=False)

print("Rolling pair summary:")
display(rolling_pair_summary)

print("Rolling universe example:")
display(rolling_universe_by_pair.head(30))


Rolling pair summary:


,pair_id,train_month,test_month,train_bin_path,test_bin_path,train_fill_path,test_fill_path,n_train_stocks,n_test_stocks,n_common_stocks_pair_universe,top20_common_stocks_by_train_notional
0,1,201901,201902,/content/project_data/processed_2_1/monthly_st...,/content/project_data/processed_2_1/monthly_st...,/content/project_data/processed_2_1/monthly_st...,/content/project_data/processed_2_1/monthly_st...,50,50,50,"[AMZN, AAPL, AMD, AMGN, ADBE, ALGN, AMAT, ANTM..."
1,2,201902,201903,/content/project_data/processed_2_1/monthly_st...,/content/project_data/processed_2_1/monthly_st...,/content/project_data/processed_2_1/monthly_st...,/content/project_data/processed_2_1/monthly_st...,50,50,50,"[AMZN, AAPL, AMD, ADBE, AMGN, AMAT, ADP, ALGN,..."
2,3,201903,201904,/content/project_data/processed_2_1/monthly_st...,/content/project_data/processed_2_1/monthly_st...,/content/project_data/processed_2_1/monthly_st...,/content/project_data/processed_2_1/monthly_st...,50,50,50,"[AMZN, AAPL, AMD, ADBE, AMGN, ALGN, ADP, ADSK,..."
3,4,201904,201905,/content/project_data/processed_2_1/monthly_st...,/content/project_data/processed_2_1/monthly_st...,/content/project_data/processed_2_1/monthly_st...,/content/project_data/processed_2_1/monthly_st...,50,50,50,"[AMZN, AAPL, AMD, ADBE, AMGN, ANTM, ALGN, APC,..."
4,5,201905,201906,/content/project_data/processed_2_1/monthly_st...,/content/project_data/processed_2_1/monthly_st...,/content/project_data/processed_2_1/monthly_st...,/content/project_data/processed_2_1/monthly_st...,50,50,50,"[AMZN, AAPL, AMD, ADBE, AMGN, APC, AMAT, ADI, ..."
5,6,201906,201907,/content/project_data/processed_2_1/monthly_st...,/content/project_data/processed_2_1/monthly_st...,/content/project_data/processed_2_1/monthly_st...,/content/project_data/processed_2_1/monthly_st...,50,50,50,"[AMZN, AAPL, AMD, ADBE, AMGN, AGN, AMAT, ADSK,..."
6,7,201907,201908,/content/project_data/processed_2_1/monthly_st...,/content/project_data/processed_2_1/monthly_st...,/content/project_data/processed_2_1/monthly_st...,/content/project_data/processed_2_1/monthly_st...,50,50,50,"[AMZN, AAPL, AMD, ADBE, AMGN, AGN, AMAT, ALGN,..."
7,8,201908,201909,/content/project_data/processed_2_1/monthly_st...,/content/project_data/processed_2_1/monthly_st...,/content/project_data/processed_2_1/monthly_st...,/content/project_data/processed_2_1/monthly_st...,50,49,49,"[AMZN, AAPL, AMD, ADBE, AMGN, ADSK, ADP, AMAT,..."
8,9,201909,201910,/content/project_data/processed_2_1/monthly_st...,/content/project_data/processed_2_1/monthly_st...,/content/project_data/processed_2_1/monthly_st...,/content/project_data/processed_2_1/monthly_st...,49,49,49,"[AMZN, AAPL, AMD, ADBE, AMGN, ADP, ADSK, AMAT,..."
9,10,201910,201911,/content/project_data/processed_2_1/monthly_st...,/content/project_data/processed_2_1/monthly_st...,/content/project_data/processed_2_1/monthly_st...,/content/project_data/processed_2_1/monthly_st...,49,49,49,"[AMZN, AAPL, AMD, ADBE, AMGN, AMAT, ADP, ANTM,..."


Rolling universe example:


,stock,train_total_abs_notional,train_n_rows,train_n_trading_days,test_total_abs_notional,test_n_rows,test_n_trading_days,pair_id,train_month,test_month
0,AMZN,2.185694e+10,46814,21,1.408750e+10,44284,19,1,201901,201902
1,AAPL,9.531554e+09,46860,21,6.306262e+09,44460,19,1,201901,201902
2,AMD,3.953044e+09,46858,21,3.038560e+09,44460,19,1,201901,201902
3,AMGN,2.225017e+09,46459,21,1.654364e+09,43637,19,1,201901,201902
4,ADBE,2.067497e+09,46392,21,1.802948e+09,43761,19,1,201901,201902
5,ALGN,1.139455e+09,42508,21,1.006597e+09,37659,19,1,201901,201902
6,AMAT,9.931631e+08,46844,21,1.020720e+09,44460,19,1,201901,201902
7,ANTM,8.661784e+08,42647,21,8.891198e+08,38703,19,1,201901,201902
8,AAL,8.600217e+08,46782,21,6.948527e+08,44333,19,1,201901,201902
9,ADP,8.534474e+08,45534,21,1.009731e+09,43118,19,1,201901,201902


In [27]:
# Also compute the strict global common universe: stocks present in every bin month.
# This is not the default rolling universe, but it is useful as a stricter robustness check.
sets_by_month = {
    m: set(stats.loc[stats["month"] == m, "stock"].astype(str))
    for m in bin_months
}

global_common_universe = sorted(set.intersection(*sets_by_month.values())) if sets_by_month else []

global_common_df = pd.DataFrame({"stock": global_common_universe})
global_common_df.to_csv(ROLLING_DIR / "global_common_universe_all_months.csv", index=False)

print(f"Number of stocks present in every bin month: {len(global_common_universe)}")
display(global_common_df.head(50))


Number of stocks present in every bin month: 49


,stock
0,A
1,AAL
2,AAP
3,AAPL
4,ABBV
5,ABC
6,ABMD
7,ABT
8,ACN
9,ADBE


In [28]:
# Save an overall rolling config for later sections.
rolling_config = {
    "section": "2.1 Data Preparation rolling enhancement",
    "rolling_rule": "For each consecutive month pair, use train_month as in-sample and test_month as out-of-sample.",
    "universe_rule": "Use all stocks common to the train and test month of that pair.",
    "data_storage_rule": "Use standardised monthly Parquet files plus rolling manifests; do not duplicate full rolling datasets.",
    "rolling_pair_summary_path": str(ROLLING_DIR / "rolling_pair_summary.csv"),
    "rolling_universe_by_pair_path": str(ROLLING_DIR / "rolling_universe_by_pair.csv"),
    "monthly_bin_parquet_dir": str(MONTHLY_DIR / "bin"),
    "monthly_fill_parquet_dir": str(MONTHLY_DIR / "fills"),
}

with open(ROLLING_DIR / "section_2_1_rolling_config.json", "w") as f:
    json.dump(rolling_config, f, indent=2)

print(json.dumps(rolling_config, indent=2))


{
  "section": "2.1 Data Preparation rolling enhancement",
  "rolling_rule": "For each consecutive month pair, use train_month as in-sample and test_month as out-of-sample.",
  "universe_rule": "Use all stocks common to the train and test month of that pair.",
  "data_storage_rule": "Use standardised monthly Parquet files plus rolling manifests; do not duplicate full rolling datasets.",
  "rolling_pair_summary_path": "/content/project_data/processed_2_1/rolling_full_universe/rolling_pair_summary.csv",
  "rolling_universe_by_pair_path": "/content/project_data/processed_2_1/rolling_full_universe/rolling_universe_by_pair.csv",
  "global_common_universe_path": "/content/project_data/processed_2_1/rolling_full_universe/global_common_universe_all_months.csv",
  "monthly_bin_parquet_dir": "/content/project_data/processed_2_1/monthly_standardised/bin",
  "monthly_fill_parquet_dir": "/content/project_data/processed_2_1/monthly_standardised/fills"
}


## 8. How later sections should load this data

The outputs below are the stepping stone from Section 2.1 to Sections 2.2 and 2.3.

For the **baseline**, later sections can directly load the 20-stock in-sample and out-of-sample Parquet files.

For the **rolling enhancement**, later sections should read `rolling_pair_summary.csv` and `rolling_universe_by_pair.csv`, then load the appropriate monthly Parquet files for each pair.


In [30]:
# Rolling loading example: first rolling pair.
pair_summary = pd.read_csv(ROLLING_DIR / "rolling_pair_summary.csv")
pair_universe = pd.read_csv(ROLLING_DIR / "rolling_universe_by_pair.csv")

first_pair = pair_summary.iloc[0]
first_pair_id = int(first_pair["pair_id"])
first_pair_stocks = pair_universe.loc[pair_universe["pair_id"] == first_pair_id, "stock"].astype(str).tolist()

rolling_train_month = str(first_pair["train_month"])
rolling_test_month = str(first_pair["test_month"])

rolling_train = load_month_subset("bin", rolling_train_month, first_pair_stocks)
rolling_test = load_month_subset("bin", rolling_test_month, first_pair_stocks)

print(f"Rolling pair {first_pair_id}: {rolling_train_month} -> {rolling_test_month}")
print("Number of stocks:", len(first_pair_stocks))
print("Rolling train shape:", rolling_train.shape)
print("Rolling test shape: ", rolling_test.shape)


Rolling pair 1: 201901 -> 201902
Number of stocks: 50
Rolling train shape: (2168002, 34)
Rolling test shape:  (2018914, 34)


## 9. OUTPUTS PRODUCED

This notebook produced:

- `raw_file_registry.csv` — raw files and sizes;
- `schema_check.csv` — confirmation of raw schemas;
- monthly standardised Parquet files for binSamples;
- `processed_monthly_summary.csv` — row counts, stocks, date ranges, output paths;
- `merge_vs_separate_decision.csv` — explicit decision on merging/separation;
- baseline 20-stock train/test Parquet files;
- `section_2_1_baseline_config.json` — exact baseline setup;
- `baseline_20stocks_summary.csv` — report table for selected stocks;
- `rolling_pair_summary.csv` — rolling train/test month pairs;
- `rolling_universe_by_pair.csv` — full common stock universe for each rolling pair;
- `global_common_universe_all_months.csv` — stricter all-month common universe;
- `section_2_1_rolling_config.json` — exact enhancement setup.


In [31]:
# Show output tree.
for root, dirs, files in os.walk(PROCESSED_BASE):
    level = root.replace(str(PROCESSED_BASE), "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{Path(root).name}/")
    subindent = "  " * (level + 1)
    for f in sorted(files)[:20]:
        print(f"{subindent}{f}")
    if len(files) > 20:
        print(f"{subindent}... {len(files)-20} more files")


processed_2_1/
  report_tables/
    bin_stock_month_stats.csv
    fill_stock_month_stats.csv
    merge_vs_separate_decision.csv
    processed_monthly_summary.csv
    raw_file_registry.csv
    schema_check.csv
  rolling_full_universe/
    global_common_universe_all_months.csv
    rolling_pair_summary.csv
    rolling_universe_by_pair.csv
    section_2_1_rolling_config.json
  monthly_standardised/
    fills/
      fills_201901_standardised.parquet
      fills_201901_stock_stats.csv
      fills_201902_standardised.parquet
      fills_201902_stock_stats.csv
      fills_201903_standardised.parquet
      fills_201903_stock_stats.csv
      fills_201904_standardised.parquet
      fills_201904_stock_stats.csv
      fills_201905_standardised.parquet
      fills_201905_stock_stats.csv
      fills_201906_standardised.parquet
      fills_201906_stock_stats.csv
      fills_201907_standardised.parquet
      fills_201907_stock_stats.csv
      fills_201908_standardised.parquet
      fills_201908_stock_s

### Saving Section 2.1 outputs for Section 2.2

In [32]:
import shutil
from pathlib import Path

DRIVE_OUTPUT_BASE = Path("/content/drive/MyDrive/Quantitative Trading and Price Impact/project_data")
DRIVE_PROCESSED_21 = DRIVE_OUTPUT_BASE / "processed_2_1"

DRIVE_PROCESSED_21.mkdir(parents=True, exist_ok=True)

shutil.copytree(
    PROCESSED_BASE,
    DRIVE_PROCESSED_21,
    dirs_exist_ok=True
)

print("Copied Section 2.1 outputs to Drive:")
print(DRIVE_PROCESSED_21)

for p in (DRIVE_PROCESSED_21 / "rolling_full_universe").glob("*"):
    print(p.name)

Copied Section 2.1 outputs to Drive:
/content/drive/MyDrive/Quantitative Trading and Price Impact/project_data/processed_2_1
global_common_universe_all_months.csv
rolling_universe_by_pair.csv
section_2_1_rolling_config.json
rolling_pair_summary.csv
